# Vehicle Model Parameter Optimization - Differential Evolution

This notebook optimizes vehicle model parameters using Differential Evolution to minimize
the difference between simulated and real trajectories across all 7 CSV files.

## Parameters
- **Global (shared across all trajectories)**: mass, Inertia_z, Inertia_tire, Inertia_engine, air_resistance, c_1y, c_2y, C_y, E_y, C_roll1, C_roll2, radius_tire
- **Per-trajectory**: mu, c_1x, c_2x, C_x, E_x (5 params × 7 trajectories = 35 params)

In [1]:
# Cell 1: Imports and Setup
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import jax
import jax.numpy as jnp
from pathlib import Path
from scipy.optimize import differential_evolution
import matplotlib.pyplot as plt
from datetime import datetime
import json
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

print(f"JAX devices: {jax.devices()}")
print(f"JAX backend: {jax.default_backend()}")

JAX devices: [CpuDevice(id=0)]
JAX backend: cpu


In [2]:
# Cell 2: Load Real Trajectory Data
data_dir = Path(r"/bigwork/nhkbarit/thesis-code/data/measurements")
csv_files = sorted(data_dir.glob("*.csv"))

print(f"Found {len(csv_files)} CSV files:")
trajectories = []
for i, f in enumerate(csv_files):
    df = pd.read_csv(f)
    # Determine surface type
    name = f.stem
    if 'asphalt' in name.lower():
        surface = 'Asphalt'
    elif 'beton' in name.lower():
        surface = 'Concrete'
    elif 'basalt' in name.lower():
        surface = 'Basalt'
    else:
        surface = 'Unknown'
    
    trajectories.append({
        'name': name,
        'df': df,
        'surface': surface,
        'path': f
    })
    print(f"  {i+1}. {name}: {len(df)} rows ({surface})")

print(f"\n✓ Loaded {len(trajectories)} trajectories")

Found 7 CSV files:
  1. Jeversen_2021_12_15_112145_Asphalt-Vollbremsung: 1070 rows (Asphalt)
  2. Jeversen_2021_12_15_112328_Asphalt-Vollbremsung: 1436 rows (Asphalt)
  3. Jeversen_2021_12_15_125650_Beton-Vollbremsung: 1189 rows (Concrete)
  4. Jeversen_2021_12_15_125936_Beton-Vollbremsung: 1263 rows (Concrete)
  5. Jeversen_2021_12_15_134709_Basalt-Vollbremsung: 1377 rows (Basalt)
  6. Jeversen_2021_12_15_134858_Basalt-Vollbremsung: 1356 rows (Basalt)
  7. Jeversen_2022_10_12_110132: 3165 rows (Unknown)

✓ Loaded 7 trajectories


In [3]:
# Cell 3: Define Parameter Structure

# Global parameters (shared across all trajectories)
GLOBAL_PARAMS = [
    'mass', 'Inertia_z', 'Inertia_tire', 'Inertia_engine',
    'air_resistance', 'c_1y', 'c_2y', 'C_y', 'E_y',
    'C_roll1', 'C_roll2', 'radius_tire'
]

# Per-trajectory parameters (different for each of 7 trajectories)
PER_TRAJ_PARAMS = ['mu', 'c_1x', 'c_2x', 'C_x', 'E_x']

NUM_TRAJECTORIES = 7
NUM_GLOBAL = len(GLOBAL_PARAMS)
NUM_PER_TRAJ = len(PER_TRAJ_PARAMS)
TOTAL_PARAMS = NUM_GLOBAL + NUM_PER_TRAJ * NUM_TRAJECTORIES

print(f"Global parameters ({NUM_GLOBAL}): {GLOBAL_PARAMS}")
print(f"Per-trajectory parameters ({NUM_PER_TRAJ}): {PER_TRAJ_PARAMS}")
print(f"Total parameters: {NUM_GLOBAL} + {NUM_PER_TRAJ} × {NUM_TRAJECTORIES} = {TOTAL_PARAMS}")

Global parameters (12): ['mass', 'Inertia_z', 'Inertia_tire', 'Inertia_engine', 'air_resistance', 'c_1y', 'c_2y', 'C_y', 'E_y', 'C_roll1', 'C_roll2', 'radius_tire']
Per-trajectory parameters (5): ['mu', 'c_1x', 'c_2x', 'C_x', 'E_x']
Total parameters: 12 + 5 × 7 = 47


In [4]:
# Cell 4: Define Parameter Bounds

# Default values from VehicleModel.py
DEFAULTS = {
    # Global params
    'mass': 1720,
    'Inertia_z': 2066,
    'Inertia_tire': 28.6,
    'Inertia_engine': 0.197,
    'air_resistance': 0.27,
    'c_1y': 1.9e6,
    'c_2y': 1.35e5,
    'C_y': 1.9,
    'E_y': 0.52,
    'C_roll1': 0.0083,
    'C_roll2': 0.0005,
    'radius_tire': 0.3116,
    # Per-trajectory params
    'mu': 0.8,
    'c_1x': 2.5e7,
    'c_2x': 3.6e6,
    'C_x': 1.42,
    'E_x': -9.75,
}

# Bounds for global parameters
GLOBAL_BOUNDS = {
    'mass': (1500, 2500),
    'Inertia_z': (1500, 3000),
    'Inertia_tire': (15, 80),
    'Inertia_engine': (0.1, 0.5),
    'air_resistance': (0.2, 1.0),
    'c_1y': (1e6, 5e6),
    'c_2y': (5e4, 5e5),
    'C_y': (1.0, 3.0),
    'E_y': (-1.0, 1.0),
    'C_roll1': (0.001, 0.05),
    'C_roll2': (0.0001, 0.01),
    'radius_tire': (0.28, 0.40),
}

# Bounds for per-trajectory parameters
PER_TRAJ_BOUNDS = {
    'mu': (0.3, 1.2),  # Friction coefficient varies by surface
    'c_1x': (1e6, 1e8),
    'c_2x': (1e5, 1e7),
    'C_x': (1.0, 3.0),
    'E_x': (-15.0, 0.0),
}

# Build bounds list for optimizer
bounds = []
param_names = []

# Global params first
for p in GLOBAL_PARAMS:
    bounds.append(GLOBAL_BOUNDS[p])
    param_names.append(p)

# Then per-trajectory params
for traj_idx in range(NUM_TRAJECTORIES):
    for p in PER_TRAJ_PARAMS:
        bounds.append(PER_TRAJ_BOUNDS[p])
        param_names.append(f"{p}_traj{traj_idx+1}")

print(f"Total bounds: {len(bounds)}")
print(f"\nGlobal bounds:")
for i, p in enumerate(GLOBAL_PARAMS):
    print(f"  {p}: {bounds[i]}")
print(f"\nPer-trajectory bounds (same for all 7):")
for p in PER_TRAJ_PARAMS:
    print(f"  {p}: {PER_TRAJ_BOUNDS[p]}")

Total bounds: 47

Global bounds:
  mass: (1500, 2500)
  Inertia_z: (1500, 3000)
  Inertia_tire: (15, 80)
  Inertia_engine: (0.1, 0.5)
  air_resistance: (0.2, 1.0)
  c_1y: (1000000.0, 5000000.0)
  c_2y: (50000.0, 500000.0)
  C_y: (1.0, 3.0)
  E_y: (-1.0, 1.0)
  C_roll1: (0.001, 0.05)
  C_roll2: (0.0001, 0.01)
  radius_tire: (0.28, 0.4)

Per-trajectory bounds (same for all 7):
  mu: (0.3, 1.2)
  c_1x: (1000000.0, 100000000.0)
  c_2x: (100000.0, 10000000.0)
  C_x: (1.0, 3.0)
  E_x: (-15.0, 0.0)


In [5]:
# Cell 5: Data Preparation Functions

def prepare_real_data(df, start_idx=50, T_seg=None):
    """
    Prepare real trajectory data for comparison.
    Returns observations and controls.
    """
    if T_seg is None:
        T_seg = len(df) - start_idx
    
    T_seg = min(T_seg, len(df) - start_idx)
    
    # Extract observations (9 dims matching vehicle_fy output)
    # [yaw_rate, v_body_x, v_body_y, a_body_x, a_body_y, tire_fl, tire_fr, tire_rl, tire_rr]
    y_real = np.zeros((T_seg, 9), dtype=np.float32)
    
    # Yaw rate (deg/s -> rad/s)
    y_real[:, 0] = df['Rate_Body_Z'].values[start_idx:start_idx+T_seg] * np.pi / 180.0
    
    # Velocities (already in m/s in typical format)
    y_real[:, 1] = df['INS_Vel_Body_X'].values[start_idx:start_idx+T_seg]
    y_real[:, 2] = df['INS_Vel_Body_Y'].values[start_idx:start_idx+T_seg]
    
    # Accelerations
    y_real[:, 3] = df['Acc_Body_X'].values[start_idx:start_idx+T_seg]
    y_real[:, 4] = df['Acc_Body_Y'].values[start_idx:start_idx+T_seg]
    
    # Tire rates (rad/s)
    y_real[:, 5] = df['Tire_Rate_FL'].values[start_idx:start_idx+T_seg]
    y_real[:, 6] = df['Tire_Rate_FR'].values[start_idx:start_idx+T_seg]
    y_real[:, 7] = df['Tire_Rate_RL'].values[start_idx:start_idx+T_seg]
    y_real[:, 8] = df['Tire_Rate_RR'].values[start_idx:start_idx+T_seg]
    
    # Extract controls (steer angle: deg -> rad to match model expectation)
    controls = {
        'steer_ang': jnp.array(np.deg2rad(df['Steer_Angle'].values[start_idx:start_idx+T_seg]), dtype=jnp.float32),
        'engine_torque': jnp.array(df['Engine_Torque'].values[start_idx:start_idx+T_seg], dtype=jnp.float32),
        'break_torque': jnp.array(df['Break_Pressure'].values[start_idx:start_idx+T_seg], dtype=jnp.float32),
        'gear_transmission': jnp.array(df['Gear_Transmission'].values[start_idx:start_idx+T_seg], dtype=jnp.float32),
    }
    
    return y_real, controls

# Prepare all trajectories
T_SEG = 800  # Use 800 timesteps for optimization (8 seconds at 100Hz)
START_IDX = 50

prepared_data = []
for traj in trajectories:
    y_real, controls = prepare_real_data(traj['df'], start_idx=START_IDX, T_seg=T_SEG)
    prepared_data.append({
        'name': traj['name'],
        'surface': traj['surface'],
        'y_real': y_real,
        'controls': controls,
        'T': y_real.shape[0]
    })
    print(f"{traj['name']}: T={y_real.shape[0]}, v_x range=[{y_real[:, 1].min():.2f}, {y_real[:, 1].max():.2f}]")

print(f"\n✓ Prepared {len(prepared_data)} trajectories")

Jeversen_2021_12_15_112145_Asphalt-Vollbremsung: T=800, v_x range=[-0.38, 23.55]
Jeversen_2021_12_15_112328_Asphalt-Vollbremsung: T=800, v_x range=[1.99, 23.53]
Jeversen_2021_12_15_125650_Beton-Vollbremsung: T=800, v_x range=[-0.37, 23.58]
Jeversen_2021_12_15_125936_Beton-Vollbremsung: T=800, v_x range=[1.00, 23.64]
Jeversen_2021_12_15_134709_Basalt-Vollbremsung: T=800, v_x range=[4.25, 23.55]
Jeversen_2021_12_15_134858_Basalt-Vollbremsung: T=800, v_x range=[4.87, 23.56]
Jeversen_2022_10_12_110132: T=800, v_x range=[11.75, 14.15]

✓ Prepared 7 trajectories


In [6]:
# Cell 6: Vehicle Model Simulation Function

from simulation.VehicleModel import (
    vehicle_RK4x, vehicle_fy,
    width_front, width_rear, length_front, length_rear,
    height_CoG, vel_limit, gravitation, air_density, body_surface_x
)

def simulate_trajectory(global_params, traj_params, controls, y0, T):
    """
    Simulate a trajectory with given parameters.
    
    Args:
        global_params: dict of global parameters
        traj_params: dict of trajectory-specific parameters (mu, c_1x, c_2x, C_x, E_x)
        controls: dict of control inputs
        y0: initial observation (9,)
        T: number of timesteps
    
    Returns:
        y_sim: simulated observations (T, 9)
    """
    # Combine parameters
    params = {**global_params, **traj_params}
    params['dt'] = 0.01
    params['radius_tire'] = global_params.get('radius_tire', 0.3116)
    
    # Build p_inf vector: [friction, air_resistance, mass]
    p_inf = jnp.array([traj_params['mu'], global_params['air_resistance'], global_params['mass']], dtype=jnp.float32)
    
    # Initial state from y0
    # State: [geo_x, geo_y, yaw, yaw_rate, v_x, v_y, tire_fl, tire_fr, tire_rl, tire_rr]
    state = jnp.array([
        0.0,  # geo_x
        0.0,  # geo_y
        0.0,  # yaw
        y0[0],  # yaw_rate
        y0[1],  # v_body_x
        y0[2],  # v_body_y
        y0[5],  # tire_fl
        y0[6],  # tire_fr
        y0[7],  # tire_rl
        y0[8],  # tire_rr
    ], dtype=jnp.float32)
    # Simulate (radius_tire is now passed through params dict)
    y_sim = []
    for t in range(T):
        u_t = {k: v[t] for k, v in controls.items()}
        y_t = vehicle_fy(state, u_t, p_inf, **params)
        y_sim.append(y_t)
        state = vehicle_RK4x(state, u_t, p_inf, **params)
    
    return jnp.stack(y_sim)

# JIT compile for speed
@jax.jit
def simulate_trajectory_jit(p_inf, state0, controls_arr, params_arr, timesteps, radius_tire):
    """
    JIT-compiled simulation.
    params_arr: [c_1x, c_2x, c_1y, c_2y, C_x, C_y, E_x, E_y, C_roll1, C_roll2,
                 mass, Inertia_z, Inertia_tire, Inertia_engine, air_resistance]
    timesteps: pre-built jnp.arange(T) array (must be concrete)
    """
    params = {
        'c_1x': params_arr[0],
        'c_2x': params_arr[1],
        'c_1y': params_arr[2],
        'c_2y': params_arr[3],
        'C_x': params_arr[4],
        'C_y': params_arr[5],
        'E_x': params_arr[6],
        'E_y': params_arr[7],
        'C_roll1': params_arr[8],
        'C_roll2': params_arr[9],
        'mass': params_arr[10],
        'Inertia_z': params_arr[11],
        'Inertia_tire': params_arr[12],
        'Inertia_engine': params_arr[13],
        'air_resistance': params_arr[14],
        'radius_tire': radius_tire,
        'dt': 0.01,
    }
    
    def body_fn(state, t):
        u_t = {
            'steer_ang': controls_arr[t, 0],
            'engine_torque': controls_arr[t, 1],
            'break_torque': controls_arr[t, 2],
            'gear_transmission': controls_arr[t, 3],
        }
        y_t = vehicle_fy(state, u_t, p_inf, **params)
        state_next = vehicle_RK4x(state, u_t, p_inf, **params)
        return state_next, y_t
    
    _, y_seq = jax.lax.scan(body_fn, state0, timesteps)
    return y_seq

print("✓ Simulation functions defined")

✓ Simulation functions defined


In [7]:
# Cell 7: Objective Function

def decode_params(x):
    """
    Decode parameter vector into global and per-trajectory dicts.
    
    x layout:
    [0:12] - global params
    [12:17] - traj 1 params (mu, c_1x, c_2x, C_x, E_x)
    [17:22] - traj 2 params
    ...
    [42:47] - traj 7 params
    """
    global_dict = {}
    for i, p in enumerate(GLOBAL_PARAMS):
        global_dict[p] = x[i]
    
    traj_params_list = []
    for traj_idx in range(NUM_TRAJECTORIES):
        start = NUM_GLOBAL + traj_idx * NUM_PER_TRAJ
        traj_dict = {}
        for j, p in enumerate(PER_TRAJ_PARAMS):
            traj_dict[p] = x[start + j]
        traj_params_list.append(traj_dict)
    
    return global_dict, traj_params_list


def objective_function(x):
    """
    Compute total MSE across all trajectories.
    """
    global_params, traj_params_list = decode_params(x)
    
    total_error = 0.0
    
    for traj_idx, data in enumerate(prepared_data):
        traj_params = traj_params_list[traj_idx]
        
        # Build parameter arrays for JIT simulation
        p_inf = jnp.array([
            traj_params['mu'],
            global_params['air_resistance'],
            global_params['mass']
        ], dtype=jnp.float32)
        
        params_arr = jnp.array([
            traj_params['c_1x'],
            traj_params['c_2x'],
            global_params['c_1y'],
            global_params['c_2y'],
            traj_params['C_x'],
            global_params['C_y'],
            traj_params['E_x'],
            global_params['E_y'],
            global_params['C_roll1'],
            global_params['C_roll2'],
            global_params['mass'],
            global_params['Inertia_z'],
            global_params['Inertia_tire'],
            global_params['Inertia_engine'],
            global_params['air_resistance'],
        ], dtype=jnp.float32)
        
        # Initial state
        y0 = data['y_real'][0]
        state0 = jnp.array([
            0.0, 0.0, 0.0,  # geo_x, geo_y, yaw
            y0[0], y0[1], y0[2],  # yaw_rate, v_x, v_y
            y0[5], y0[6], y0[7], y0[8]  # tire rates
        ], dtype=jnp.float32)
        
        # Controls array
        controls = data['controls']
        T = data['T']
        controls_arr = jnp.stack([
            controls['steer_ang'],
            controls['engine_torque'],
            controls['break_torque'],
            controls['gear_transmission']
        ], axis=1)
        
        try:
            # Simulate
            timesteps = jnp.arange(T)
            y_sim = simulate_trajectory_jit(p_inf, state0, controls_arr, params_arr, timesteps, global_params['radius_tire'])
            y_sim = np.asarray(y_sim)
            
            # Check for NaN
            if np.any(np.isnan(y_sim)):
                return 1e10
            
            # Compute weighted MSE
            y_real = data['y_real']
            
            # Weight different observation dimensions
            weights = np.array([1.0, 5.0, 1.0, 2.0, 1.0, 3.0, 3.0, 3.0, 3.0])  # More weight on v_body_x and tire rates
            
            mse = np.mean(weights * (y_sim - y_real)**2)
            total_error += mse
            
        except Exception as e:
            return 1e10
    
    return total_error / len(prepared_data)

# Test objective function
x0 = []
for p in GLOBAL_PARAMS:
    x0.append(DEFAULTS[p])
for _ in range(NUM_TRAJECTORIES):
    for p in PER_TRAJ_PARAMS:
        x0.append(DEFAULTS[p])

print(f"Testing objective with default params...")
initial_error = objective_function(x0)
print(f"Initial error (default params): {initial_error:.4f}")

Testing objective with default params...
Initial error (default params): 582.0314


In [8]:
# Cell 8: Progress Callback

class OptimizationCallback:
    def __init__(self):
        self.iteration = 0
        self.best_error = float('inf')
        self.history = []
        self.start_time = None
    
    def __call__(self, xk, convergence=None):
        self.iteration += 1
        error = objective_function(xk)
        
        if error < self.best_error:
            self.best_error = error
        
        self.history.append(error)
        
        if self.iteration % 10 == 0:
            elapsed = datetime.now() - self.start_time
            print(f"Iter {self.iteration}: Current={error:.4f}, Best={self.best_error:.4f}, Time={elapsed}")

callback = OptimizationCallback()
print("✓ Callback defined")

✓ Callback defined


In [ ]:
# Cell 9: Run Differential Evolution

print("="*70)
print("Starting Differential Evolution Optimization")
print("="*70)
print(f"Parameters: {TOTAL_PARAMS}")
print(f"Trajectories: {NUM_TRAJECTORIES}")
print(f"Population size: 15 × params = {15 * TOTAL_PARAMS}")
print("="*70)

callback.start_time = datetime.now()

result = differential_evolution(
    objective_function,
    bounds,
    strategy='best1bin',
    maxiter=200,
    popsize=15,
    tol=1e-6,
    mutation=(0.5, 1.0),
    recombination=0.7,
    seed=42,
    callback=callback,
    disp=True,
    polish=True,
    workers=1,  # JAX doesn't play well with multiprocessing
    updating='deferred',
)

print("\n" + "="*70)
print("Optimization Complete!")
print("="*70)
print(f"Success: {result.success}")
print(f"Message: {result.message}")
print(f"Final error: {result.fun:.6f}")
print(f"Iterations: {result.nit}")
print(f"Function evaluations: {result.nfev}")

Starting Differential Evolution Optimization
Parameters: 47
Trajectories: 7
Population size: 15 × params = 705
differential_evolution step 1: f(x)= 362.12165293875665
differential_evolution step 2: f(x)= 330.5299488266087
differential_evolution step 3: f(x)= 319.7321665013595
differential_evolution step 4: f(x)= 308.0729419832288
differential_evolution step 5: f(x)= 296.80834866986277
differential_evolution step 6: f(x)= 268.84697923726225
differential_evolution step 7: f(x)= 268.84697923726225
differential_evolution step 8: f(x)= 257.0254227107756
differential_evolution step 9: f(x)= 254.7050842060685
differential_evolution step 10: f(x)= 254.7050842060685
Iter 10: Current=254.7051, Best=254.7051, Time=0:58:36.636350
differential_evolution step 11: f(x)= 254.7050842060685
differential_evolution step 12: f(x)= 254.7050842060685
differential_evolution step 13: f(x)= 254.7050842060685
differential_evolution step 14: f(x)= 254.7050842060685
differential_evolution step 15: f(x)= 246.300287

In [ ]:
# Cell 10: Extract and Display Results

global_opt, traj_opt_list = decode_params(result.x)

print("="*70)
print("OPTIMIZED GLOBAL PARAMETERS")
print("="*70)
for p in GLOBAL_PARAMS:
    default = DEFAULTS[p]
    opt = global_opt[p]
    change = (opt - default) / default * 100
    print(f"  {p:20s}: {opt:12.4f} (default: {default:12.4f}, change: {change:+.1f}%)")

print("\n" + "="*70)
print("OPTIMIZED PER-TRAJECTORY PARAMETERS")
print("="*70)
for traj_idx, (data, traj_params) in enumerate(zip(prepared_data, traj_opt_list)):
    print(f"\nTrajectory {traj_idx+1}: {data['name']} ({data['surface']})")
    for p in PER_TRAJ_PARAMS:
        default = DEFAULTS[p]
        opt = traj_params[p]
        print(f"    {p:10s}: {opt:12.4f}")

In [ ]:
# Cell 11: Plot Convergence

plt.figure(figsize=(10, 4))
plt.semilogy(callback.history)
plt.xlabel('Iteration')
plt.ylabel('Error (log scale)')
plt.title('Differential Evolution Convergence')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('experiments/de_convergence.png', dpi=150)
plt.show()
print("✓ Saved convergence plot")

In [ ]:
# Cell 12: Compare Simulated vs Real for Each Trajectory

fig, axes = plt.subplots(4, 2, figsize=(14, 12))
axes = axes.flatten()

for traj_idx, (data, traj_params) in enumerate(zip(prepared_data, traj_opt_list)):
    if traj_idx >= 7:
        break
    
    ax = axes[traj_idx]
    
    # Simulate with optimized params
    p_inf = jnp.array([traj_params['mu'], global_opt['air_resistance'], global_opt['mass']], dtype=jnp.float32)
    params_arr = jnp.array([
        traj_params['c_1x'], traj_params['c_2x'],
        global_opt['c_1y'], global_opt['c_2y'],
        traj_params['C_x'], global_opt['C_y'],
        traj_params['E_x'], global_opt['E_y'],
        global_opt['C_roll1'], global_opt['C_roll2'],
        global_opt['mass'], global_opt['Inertia_z'],
        global_opt['Inertia_tire'], global_opt['Inertia_engine'],
        global_opt['air_resistance'],
    ], dtype=jnp.float32)
    
    y0 = data['y_real'][0]
    state0 = jnp.array([0.0, 0.0, 0.0, y0[0], y0[1], y0[2], y0[5], y0[6], y0[7], y0[8]], dtype=jnp.float32)
    controls = data['controls']
    T = data['T']
    controls_arr = jnp.stack([controls['steer_ang'], controls['engine_torque'],
                              controls['break_torque'], controls['gear_transmission']], axis=1)
    
    timesteps = jnp.arange(T)
    y_sim = simulate_trajectory_jit(p_inf, state0, controls_arr, params_arr, timesteps, global_opt['radius_tire'])
    y_sim = np.asarray(y_sim)
    
    t = np.arange(T) * 0.01
    ax.plot(t, data['y_real'][:, 1], 'b-', label='Real v_body_x', alpha=0.8)
    ax.plot(t, y_sim[:, 1], 'r--', label='Sim v_body_x', alpha=0.8)
    ax.set_xlabel('Time [s]')
    ax.set_ylabel('v_body_x [m/s]')
    ax.set_title(f"{data['surface']} (μ={traj_params['mu']:.3f})")
    ax.legend(loc='upper right', fontsize=8)
    ax.grid(True, alpha=0.3)

# Hide unused subplot
axes[7].axis('off')

plt.tight_layout()
plt.savefig('experiments/de_trajectory_comparison.png', dpi=150)
plt.show()
print("✓ Saved trajectory comparison plot")

In [ ]:
# Cell 13: Save Results

results_dict = {
    'method': 'Differential Evolution',
    'timestamp': datetime.now().isoformat(),
    'final_error': float(result.fun),
    'iterations': result.nit,
    'function_evaluations': result.nfev,
    'success': result.success,
    'global_params': {k: float(v) for k, v in global_opt.items()},
    'trajectory_params': [
        {
            'trajectory': data['name'],
            'surface': data['surface'],
            'params': {k: float(v) for k, v in traj_params.items()}
        }
        for data, traj_params in zip(prepared_data, traj_opt_list)
    ],
    'convergence_history': [float(e) for e in callback.history],
}

output_path = Path('experiments/de_optimization_results.json')
output_path.parent.mkdir(parents=True, exist_ok=True)
with open(output_path, 'w') as f:
    json.dump(results_dict, f, indent=2)

print(f"✓ Results saved to {output_path}")

In [ ]:
# Cell 14: Summary Statistics

print("="*70)
print("FRICTION COEFFICIENTS BY SURFACE")
print("="*70)

# Group by surface
surface_mu = {}
for data, traj_params in zip(prepared_data, traj_opt_list):
    surface = data['surface']
    if surface not in surface_mu:
        surface_mu[surface] = []
    surface_mu[surface].append(traj_params['mu'])

for surface, mus in surface_mu.items():
    print(f"{surface}: μ = {np.mean(mus):.4f} ± {np.std(mus):.4f} (n={len(mus)})")

print("\n" + "="*70)
print("OPTIMIZATION COMPLETE")
print("="*70)